# MMM deep-dive — Bayesian (PyMC-Marketing) vs Ridge fallback

The Streamlit app ships with the **Ridge / NNLS** MMM (`src/mmm.py`) because Streamlit Community Cloud's build window can't accommodate the PyMC compile chain. This notebook is the Bayesian-first version — same DGP, same data, but with proper uncertainty quantification.

**Requirements**: `pip install pymc-marketing>=0.7`. Skip the PyMC sections if it's not installed; the closing comparison still runs against the Ridge fit.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from src.data_generation import CHANNELS, generate_dataset
from src.mmm import MMM

df, truth = generate_dataset(seed=42)
df.head(3)

## 1. Ridge baseline (what ships in the app)

In [ ]:
ridge = MMM().fit(df)
print(f'Ridge R² = {ridge.r_squared:.3f}, MAPE = {ridge.mape:.2%}')

## 2. Bayesian MMM with PyMC-Marketing

Only run if `pymc-marketing` is installed. We use:

- Geometric adstock with `Beta(2, 2)` prior on λ (centred on 0.5, supports the [0,1] domain).
- Hill saturation with `LogNormal` priors on `α` and `λ`.
- `HalfNormal(2)` prior on channel coefficients (mirrors NNLS non-negativity).
- Standardised target so the intercept absorbs the scale.

In [ ]:
try:
    from pymc_marketing.mmm import MMM as PyMCMMM
    from pymc_marketing.mmm.transformers import GeometricAdstock, HillSaturation
    have_pymc = True
except ImportError:
    print('pymc-marketing not installed — skipping Bayesian section.')
    have_pymc = False

In [ ]:
if have_pymc:
    channel_cols = [c.name for c in CHANNELS]
    control_cols = ['is_weekend', 'is_holiday', 'temperature_c', 'promo_pct']

    pymc_mmm = PyMCMMM(
        date_column='date',
        channel_columns=channel_cols,
        control_columns=control_cols,
        adstock=GeometricAdstock(l_max=21),
        saturation=HillSaturation(),
        # Defaults for priors are sensible; override here if needed.
    )
    pymc_mmm.fit(
        X=df[['date', *channel_cols, *control_cols]],
        y=df['units_sold'],
        chains=2, draws=1000, tune=500, target_accept=0.9, random_seed=42,
    )
    pymc_mmm.summary()

In [ ]:
if have_pymc:
    posterior = pymc_mmm.fit_result
    # Extract posterior means for channel coefficients (β)
    bayes_betas = (
        posterior.posterior['beta_channel'].mean(dim=['chain', 'draw']).to_pandas()
    )
    bayes_betas

## 3. Side-by-side comparison

In [ ]:
rows = []
for c in CHANNELS:
    rows.append({
        'channel':     c.name,
        'true β':      truth.betas[c.name],
        'Ridge β':     ridge.betas[c.name],
        'Bayesian β':  float(bayes_betas[c.name]) if have_pymc else float('nan'),
    })
comp = pd.DataFrame(rows).set_index('channel')
comp.style.format('{:,.0f}')

## 4. Why Bayesian for production?

- **Uncertainty intervals** on every coefficient — "search ROI is 4.1× ± 0.6" beats "4.1×".
- **Informative priors** on λ and α regularise the model when channel histories are short or correlated.
- **Posterior predictive checks** — easy to validate the model captures the seasonality before trusting the channel attributions.
- **Hierarchical extensions** — same machinery scales to geo-MMM with shared global priors and per-region deviations.

The Ridge version stays useful as a 5-second sanity check and as the production fallback when build constraints don't allow PyMC.